# Engenharia de Atributos Aplicada à Modelagem Macroeconômica

Este módulo documenta o processo de construção de variáveis derivadas
a partir das séries históricas e projetadas do IPCA e da SELIC,
com o propósito de enriquecer a base analítica utilizada pelo
Agente Macroeconômico desenvolvido neste estudo.

Diferentemente da utilização direta de séries em nível,
a Engenharia de Atributos permite capturar:

- Dinâmicas inflacionárias intertemporais;
- Estrutura de juros reais ex-ante;
- Mudanças de regime monetário;
- Padrões estruturais de médio e longo prazo.

Em termos metodológicos, essa etapa amplia o conteúdo informacional
das variáveis originais, possibilitando maior capacidade de generalização
e interpretação econômica dos resultados.

A abordagem adotada reconhece que, em sistemas macroeconômicos,
a informação relevante frequentemente não está no valor absoluto
das variáveis, mas em suas variações, interações e componentes estruturais.

## Transformando o JSON em DataFrame

In [ ]:
# Transforma cada histórico em um DataFrame e adiciona a coluna 'indicador'
"""
Nesta etapa, os dados históricos retornados pela API são convertidos
em estruturas tabulares independentes (DataFrames) para cada indicador.
"""
# Conversão do histórico do IPCA para DataFrame estruturado
df_ipca = pd.DataFrame(resultados['IPCA']['historico'])
# Identificação explícita do indicador (variável categórica)
df_ipca['indicador'] = 'IPCA'

# Conversão do histórico da SELIC para DataFrame estruturado
df_selic = pd.DataFrame(resultados['SELIC']['historico'])
# Identificação explícita do indicador
df_selic['indicador'] = 'SELIC'

In [ ]:
# Consolidação Estrutural das Séries Macroeconômicas
"""
Nesta etapa, realiza-se a unificação das séries de IPCA e SELIC
em um único DataFrame com estrutura padronizada.

A consolidação não implica mistura estatística das séries,
mas apenas padronização organizacional.
"""
# União vertical das séries (empilhamento)
# ignore_index=True garante reconstrução sequencial do índice
df_historico = pd.concat([df_ipca, df_selic], ignore_index=True)


"""
Converte a coluna 'data' para datetime.
Sem essa conversão, a coluna seria tratada como string,
comprometendo análises temporais posteriores.
"""
df_historico['data'] = pd.to_datetime(df_historico['data'])


## Limpeza e ajustes

In [ ]:
"""
Nesta etapa realiza-se o tratamento de valores ausentes e a
eliminação de registros duplicados, assegurando integridade
e coerência temporal das séries.

Justificativa metodológica:
---------------------------
Séries macroeconômicas podem apresentar lacunas decorrentes de:
- Feriados bancários (no caso da SELIC diária);
- Ausência de publicação em determinados períodos;
- Problemas na coleta automatizada via API.

Para preservar a continuidade temporal sem distorcer a dinâmica
intra-série, aplica-se o método Forward Fill (ffill),
que propaga o último valor observado.

Importante:
-----------
O preenchimento é realizado separadamente por indicador
(groupby), evitando contaminação cruzada entre IPCA e SELIC.
"""

df_historico['valor'] = (
    df_historico
    .sort_values(['indicador', 'data'])   # garante ordem cronológica correta
    .groupby('indicador')['valor']        # separa as séries
    .ffill()                              # propaga último valor válido
)

"""
Remoção de registros duplicados.

A duplicidade pode ocorrer em processos de concatenação
ou múltiplas chamadas à API.

A combinação (data, indicador) define uma observação única,
portanto qualquer repetição dessa chave é eliminada.
"""
df_historico = df_historico.drop_duplicates(subset=['data', 'indicador'])  # remove duplicatas

## Padronização de datas 

In [ ]:
# Re-segmentação das Séries por Indicador
"""
Embora a padronização estrutural tenha sido realizada de forma unificada,
IPCA e SELIC apresentam naturezas econômicas e propriedades estatísticas distintas.

O IPCA consiste em uma série mensal de inflação,
caracterizada por comportamento sazonal recorrente,
especialmente influenciado por padrões de consumo,
choques de oferta e ciclos econômicos.

Já a SELIC, originalmente registrada em frequência diária,
reflete decisões de política monetária e tende a apresentar
dinâmica menos sazonal e mais associada a regimes de juros.

Dessa forma, as etapas de modelagem e projeção exigem
tratamentos diferenciados:

- Para o IPCA, será aplicado o modelo SARIMA,
  adequado para séries com componente sazonal explícito;

- Para a SELIC, será utilizado o modelo ARIMA,
  considerando sua dinâmica autoregressiva e ausência
  de sazonalidade estrutural significativa após agregação mensal.

A separação das séries garante coerência metodológica
na especificação dos modelos econométricos,
evitando imposição inadequada de estrutura sazonal
onde ela não se verifica empiricamente.
"""
df_ipca = df_historico[df_historico['indicador'] == 'IPCA'].copy()
df_selic = df_historico[df_historico['indicador'] == 'SELIC'].copy()

# Padronização do Índice Temporal
# Conversão explícita para datetime (garantia de tipagem correta)
df_selic['data'] = pd.to_datetime(df_selic['data'])
df_selic = df_selic.set_index('data')

df_ipca['data'] = pd.to_datetime(df_ipca['data'])
df_ipca = df_ipca.set_index('data')

### Transformando SELIC em dado mensalizado 

In [ ]:
# Agregação da SELIC para Frequência Mensal
"""
A série da SELIC é originalmente disponibilizada em frequência diária.
Para compatibilização com o IPCA (mensal) e viabilização de análises
conjuntas e projeções macroeconômicas, realiza-se a agregação temporal.

Critério adotado:
Utiliza-se o último valor observado em cada mês (Month End).

Justificativa econômica:
A taxa SELIC representa o instrumento operacional da política monetária.
O último valor do mês reflete a taxa vigente ao encerramento do período,
sendo mais representativo para análises de regime monetário e
comparações com indicadores mensais de inflação.

Alternativas possíveis (não adotadas):
- Média mensal → dilui decisões discretas do COPOM;
- Mediana → pouco interpretável no contexto de política monetária.

Assim, a escolha pelo último valor preserva a lógica decisória
da autoridade monetária e mantém coerência econômica.
"""
df_selic_mensal = (
    df_selic
    .resample('ME')   # Agregação para fim do mês
    .last()           # Última taxa vigente no período
)

# Padronização do Índice Temporal
"""
Após a agregação, ajusta-se o índice temporal para o primeiro dia do mês.

Motivação técnica:
- Padronizar alinhamento temporal com o IPCA;
- Evitar ambiguidade entre fim e início do período;
- Facilitar merges e modelagens futuras.

A transformação mantém a referência mensal,
mas desloca a marcação para o início do período.
"""
df_selic_mensal.index = df_selic_mensal.index.to_period('M').to_timestamp()

## MODELO SARIMA IPCA

In [ ]:
print("\n" + "="*165)
print("\033[1m" + "IPCA".center(150) + "\033[0m")
print("="*165 + "\n")

In [ ]:
# Converte o índice para o tipo DatetimeIndex do pandas.
df_ipca.index = pd.to_datetime(df_ipca.index)
df_ipca = df_ipca.asfreq('MS')   # MS = Month Start


In [ ]:
# Preparação da Série Temporal
serie = df_ipca['valor']

# Teste de Estacionariedade (ADF)
"""
Antes da estimação do modelo ARIMA, verifica-se
a presença de raiz unitária por meio do
Teste de Dickey-Fuller Aumentado (ADF).

Hipótese nula (H0):
A série possui raiz unitária (não estacionária).

Se o p-valor for superior a 5%, não se rejeita H0,
indicando necessidade de diferenciação (d = 1).

Caso contrário, a série pode ser modelada em nível (d = 0).
"""
teste_adf(serie)

# Estimação e Seleção do Modelo SARIMA
"""
Nesta etapa é realizado o procedimento de seleção automática
do modelo SARIMA mais adequado à série do IPCA.

A função 'ajustar_sarima_auto' testa diferentes combinações
de parâmetros (p,d,q)(P,D,Q,12) e seleciona aquela que minimiza
um critério de informação (ex.: AIC ou BIC).

Fundamentação:
Os critérios de informação penalizam a complexidade do modelo,
buscando equilíbrio entre ajuste (log-verossimilhança) e parcimônia.

Resultado:
- melhor_modelo → objeto SARIMAX já estimado
- df_modelos    → tabela com métricas comparativas dos modelos testados
"""
melhor_modelo, df_modelos = ajustar_sarima_auto(df_ipca['valor'])

# Geração de Previsões Fora da Amostra
"""
Com o modelo selecionado, realiza-se a projeção do IPCA
para os próximos 48 meses.

A previsão é do tipo dinâmica (out-of-sample), baseada
exclusivamente na estrutura estimada do modelo e na
informação disponível até o último período observado.

O horizonte de 48 passos corresponde a 4 anos à frente,
considerando periodicidade mensal.
"""
projecao_ipca = prever_ipca(melhor_modelo, passos=48)

# Visualização da Trajetória Projetada
"""
Nesta etapa é construída a representação gráfica da série
histórica e da trajetória projetada.

O gráfico permite:

1) Comparar valores observados e previstos;
2) Avaliar a coerência dinâmica da projeção;
3) Visualizar a incerteza associada às estimativas
   por meio do intervalo de confiança.

A ampliação do intervalo ao longo do horizonte
reflete o acúmulo da variância do erro de previsão.
"""
plotar_previsao(df_ipca['valor'], projecao_ipca)

## MODELO ARIMA SELIC

In [ ]:
print("\n" + "="*165)
print("\033[1m" + "SELIC".center(150) + "\033[0m")
print("="*165 + "\n")

In [ ]:
# Converte o índice para o tipo DatetimeIndex do pandas.
df_selic_mensal.index = pd.to_datetime(df_selic_mensal.index)
df_selic_mensal = df_selic_mensal.asfreq('MS')   # MS = Month Start

In [ ]:
# Preparação da Série Temporal
serie_selic = df_selic_mensal['valor']

# Teste de Estacionariedade (ADF)
"""
Antes da estimação do modelo ARIMA, verifica-se
a presença de raiz unitária por meio do
Teste de Dickey-Fuller Aumentado (ADF).

Hipótese nula (H0):
A série possui raiz unitária (não estacionária).

Se o p-valor for superior a 5%, não se rejeita H0,
indicando necessidade de diferenciação (d = 1).

Caso contrário, a série pode ser modelada em nível (d = 0).
"""
testar_estacionariedade(serie_selic)

In [ ]:
# Interpretação dos gráficos de serie original
plt.figure(figsize=(10,4))
plt.plot(serie_selic)
plt.title("Série Original da SELIC")
plt.show()
print("           Mostra que a série original não é estacionária, geralmente com tendência ou variação não constante")

# Interpretação dos gráficos de serie d=1 
plt.figure(figsize=(10,4))
plt.plot(serie_selic.diff().dropna())
plt.title("Primeira Diferença da SELIC (d=1)")
plt.show()
print("           O grafico Mostra que a primeira diferença remove a tendência, deixando a série visualmente estacionária")

# Interpretação dos gráficos de serie d=2
plt.figure(figsize=(10,4))
plt.plot(serie_selic.diff().diff().dropna())
plt.title("Segunda Diferença da SELIC (d=2)")
plt.show()
print("    Mostra que a segunda diferença costuma exagerar na remoção da tendência, causando ruído excessivo e perda de estrutura")

In [ ]:
# Interpretação dos gráficos ACF
'''
ACF da primeira diferença (d=1)
- A autocorrelação começa alta e decresce suavemente conforme os lags aumentam.
- A queda gradual indica que a série possui dependência temporal consistente e estrutura de memória.
- Os valores estão dentro da faixa de significância (área azul), sem picos abruptos fora dela.
- Isso sugere estacionariedade fraca e um padrão que pode ser modelado com ARIMA/SARIMA com d=1.
'''
plt.figure(figsize=(10,4))
plot_acf(serie_selic.diff().dropna(), lags=30)
plt.title("ACF da Primeira Diferença (d=1)")
plt.show()

'''
ACF da segunda diferença (d=2)
- pico negativo forte no lag 1 (em torno de -0.7).
- Esse comportamento é típico de over-differencing, que cria autocorrelação negativa no primeiro lag.
- Após o pico, a autocorrelação praticamente desaparece, indicando que o processo perdeu quase toda a estrutura temporal útil.
- Isso é sinal clássico de que a série foi diferenciada mais do que o necessário.
'''
plt.figure(figsize=(10,4))
plot_acf(serie_selic.diff().diff().dropna(), lags=30)
plt.title("ACF da Segunda Diferença (d=2)")
plt.show()

'''
Conclusão
- A primeira diferença (d=1) preserva a estrutura temporal natural da série, indicando que a série já está suficientemente estacionarizada.
- A segunda diferença (d=2) remove excesso de estrutura, gerando ruído e autocorrelação negativa, um sinal claro de over-differencing.
- Portanto, d = 1 é a escolha mais adequada, pois:
    - Evita over-differencing
    - Mantém informações temporais importantes
    - Facilita modelagem e previsão mais confiável
'''

In [ ]:
# Estimação e Seleção do Modelo ARIMA
"""
Nesta etapa realiza-se a seleção automática
do modelo ARIMA(p,d,q) mais adequado à série da SELIC.

A função 'ajustar_arima_auto' testa múltiplas combinações
dos parâmetros (p,q), mantendo fixo o grau de diferenciação (d),
e seleciona a especificação que minimiza o Critério
de Informação de Akaike (AIC).

O AIC penaliza a complexidade do modelo,
buscando equilíbrio entre ajuste e parcimônia.

Resultado:
- modelo_selic      → modelo estimado com menor AIC
- df_modelos_selic  → tabela comparativa dos modelos testados
"""
modelo_selic, df_modelos_selic = ajustar_arima_auto(
    serie=serie_selic,
    d=1  # utilizar 0 caso a série seja estacionária
)

# Geração de Projeções Fora da Amostra
"""
Com o modelo selecionado, realiza-se a projeção
da taxa SELIC até o ano de 2030.

A previsão é do tipo dinâmica (out-of-sample),
baseada exclusivamente na estrutura estatística
capturada pelo modelo ARIMA estimado.

Os intervalos de confiança de 95% refletem
a incerteza associada ao erro de previsão,
que tende a aumentar ao longo do horizonte projetado.
"""
projecao_selic = projetar_arima(
    modelo=modelo_selic,
    serie=serie_selic,
    ano_final=2030
)

# Visualização da Trajetória Histórica e Projetada
"""
Nesta etapa constrói-se o gráfico comparativo
entre a série histórica observada e a trajetória
projetada pelo modelo ARIMA.

O gráfico permite:

1) Avaliar a coerência dinâmica da projeção;
2) Comparar valores históricos e estimados;
3) Visualizar o intervalo de confiança;
4) Identificar o ponto de transição entre
   dados observados e projeção futura.

A ampliação progressiva do intervalo de confiança
decorre do acúmulo da variância do erro de previsão,
característica inerente a modelos ARIMA.
"""
plotar_projecao(
    df_mensal=df_selic_mensal,
    df_previsao=projecao_selic,
    titulo_modelo="ARIMA Selecionado via AIC"
)

## Junção da SELIC e IPCA

### Junção dos DataFrames Passdados Mensalizados 

In [ ]:
# Construção da Base Macroeconômica – Dados Observados
# Concatenando os dataframes passados
df_macro_observados = pd.concat([df_ipca, df_selic_mensal])

# Garante que o índice esteja no formato datetime
df_macro_observados.index = pd.to_datetime(df_macro_observados.index)

# Ordenar cronologicamente o índice
df_macro_observados = df_macro_observados.sort_index()

# Criar coluna nova com valor fixo "observado"
df_macro_observados['periodo_cenario'] = 'observado'

### Junção dos DataFrames Futuros projetados Mensalizados 

In [ ]:
# Renomeando a coluna prevista
df_selic_projecao = projecao_selic.rename(columns={'selic_prevista': 'valor'})
df_ipca_projecao = projecao_ipca.rename(columns={'ipca_previsto': 'valor'})

In [ ]:
# Adicionar coluna "indicador" com valor fixo "SELIC"
df_selic_projecao['indicador'] = 'SELIC'

# Adicionar coluna "indicador" com valor fixo "IPCA"
df_ipca_projecao['indicador'] = 'IPCA'

In [ ]:
# Concatenando os dataframes 
df_macro_projecoes = pd.concat([df_selic_projecao, df_ipca_projecao])

# Criar coluna nova com valor fixo "projetado"
df_macro_projecoes['periodo_cenario'] = 'projetado'

# Garantir que o índice é datetime
df_macro_projecoes.index = pd.to_datetime(df_macro_projecoes.index)

# Ordenar o índice
df_macro_projecoes = df_macro_projecoes.sort_index()

# colocando limite de data menores que 1º de janeiro de 2030.
df_resultado_macro_projecoes = df_macro_projecoes[df_macro_projecoes.index < '2030-01-01']

# Reorganizar as colunas na ordem
df_resultado_macro_projecoes = df_resultado_macro_projecoes[['valor', 'indicador', 'periodo_cenario', 'limite_inferior', 'limite_superior']]

### Junção dos DataFrames Futuros e Passados

In [ ]:
# Concatenando os dataframes 
df_concat_resultado_indicadores = pd.concat([df_resultado_macro_projecoes, df_macro_observados], axis=0, sort=False)

# Garantir que o índice é datetime
df_concat_resultado_indicadores.index = pd.to_datetime(df_concat_resultado_indicadores.index)

# Ordenar o índice
df_concat_resultado_indicadores = df_concat_resultado_indicadores.sort_index()

## Tranformações necessárias antes dos cálculos

### Substituindo os valores nulos das colunas limite_superior e limite_inferior por valor

In [ ]:
# Tratamento de Valores Ausentes nos Intervalos de Confiança
# defininfo nome do df
df_preench_vl_null = df_concat_resultado_indicadores

# Ajuste da coluna 'limite_inferior'
df_preench_vl_null['limite_inferior'] = df_preench_vl_null['limite_inferior'].where(
    # Mantém o valor original quando NÃO for NaN
    df_preench_vl_null['limite_inferior'].notna(),

    # Quando for NaN, substitui pelo valor da coluna 'valor'
    df_preench_vl_null['valor']
)

# Ajuste da coluna 'limite_superior'
df_preench_vl_null['limite_superior'] = df_preench_vl_null['limite_superior'].where(
    # Mantém o valor original quando NÃO for NaN
    df_preench_vl_null['limite_superior'].notna(),

    # Mantém o valor original quando NÃO for NaN
    df_preench_vl_null['valor']
)

### Pivot de algumas colunas

In [ ]:
# Reestruturação da Base para Formato de Cenários
'''
Transformar o DataFrame consolidado (com valores pontuais 
e intervalos de confiança) em um formato "long" estruturado
por cenários (base, pessimista e otimista).
'''
df_pivot_col_final = (
    df_preench_vl_null
        # Resetar índice para transformar a data em coluna explícita
        .reset_index()
    
        # Garantir nome padronizado da coluna temporal
        .rename(columns={'index': 'data'})

        # Assegurar que a coluna 'data' esteja em formato datetime
        .assign(data=lambda x: pd.to_datetime(x['data']))

        # Converte colunas: - valor - limite_inferior  - limite_superior Em uma única coluna categórica chamada 'cenarios', mantendo como identificadores: 
        # data, periodo_cenario, indicador
        .melt(
            id_vars=['data', 'periodo_cenario', 'indicador'],
            value_vars=['valor', 'limite_inferior', 'limite_superior'],
            var_name='cenarios',
            value_name='vl_taxa'
        )
        # Renomeia os valores da coluna 'cenarios'
        #   valor → cenário base (projeção central)
        #   limite_superior → cenário otimista
        #   limite_inferior → cenário pessimista
        .replace({
            'cenarios': {
                'valor': 'base',
                'limite_superior': 'otimista',
                'limite_inferior': 'pessimista'
            }
        })
        .sort_values('data')   # ordena mantendo como coluna
)
